# Monarch-inspired attention: follow-up probe battery

Eight probes from a research-advisor (Claude Fable 5) consultation on tonight's
causal MonarchAttention / SlidingMonarchAttention / MetaMonarchAttention
investigation, picked for being cheap, self-contained, and benefiting from more
scale/trials than was comfortable on local CPU. No GPU strictly required, but
select `Runtime > Change runtime type > GPU` if you want the larger sweeps to
run faster.

Run top to bottom (`Runtime > Run all`). Each probe is self-contained after the
setup cell. Copy back whichever sections' output you want folded into the
research journal.

## Setup: port of `ma_sliding_monarch.py` and `ma_meta_bucket_route.py`

In [ ]:
import torch
import torch.nn.functional as F
from math import sqrt
import math

print(f"PyTorch {torch.__version__}, CUDA available: {torch.cuda.is_available()}")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

In [ ]:
# ---- ported from ma_sliding_monarch.py ----

def _local_pass(ar, k, cr, sm_scale, mask, eps=1e-12):
    r_hat = sm_scale * (ar @ k.transpose(-1, -2)).to(ar.dtype)
    r_hat = r_hat / (cr[..., :, None] + eps)
    r_hat = r_hat + torch.where(mask, 0.0, -float("inf"))
    r_hat = torch.exp(
        r_hat - torch.clamp(torch.max(r_hat, dim=-1, keepdim=True).values, min=eps)
    )
    r = r_hat / (torch.sum(r_hat, dim=-1, keepdim=True) + eps)
    r = torch.clamp(r, min=torch.finfo(r.dtype).tiny)
    cl = torch.sum(torch.special.xlogy(r, r), dim=-1).transpose(-1, -2)
    al = sm_scale * (r.to(k.dtype) @ k).transpose(-2, -3)
    return al, r, cl


def _local_pass_with_v(ar, k, v, cr, sm_scale, mask, eps=1e-12):
    al, r, cl = _local_pass(ar, k, cr, sm_scale, mask, eps)
    y = (r.to(v.dtype) @ v).transpose(-2, -3)
    return al, y, cl


def sliding_monarch_causal(q, k, v, B, W_blocks, T, W_refine=None, eps=1e-6, return_deltas=False):
    """Same as tonight's ma_sliding_monarch.py, with an optional
    return_deltas hook (probe 1 needs per-iteration delta norms)."""
    if W_refine is None:
        W_refine = W_blocks
    E, H, N, D = q.shape
    _, _, _, Dv = v.shape
    M = (N + B - 1) // B
    N_padded = M * B
    sm_scale = 1 / sqrt(D)

    pad = N_padded - N
    qb = F.pad(q, (0, 0, 0, pad)).view(E, H, M, B, D)
    kb = F.pad(k, (0, 0, 0, pad)).view(E, H, M, B, D)
    vb = F.pad(v, (0, 0, 0, pad)).view(E, H, M, B, Dv)

    range_n = torch.arange(N_padded, device=q.device).view(M, B)
    valid_mb = (range_n < N)

    ar = qb
    cr = torch.ones(E, H, M, B, device=q.device, dtype=q.dtype)
    q_t = qb.transpose(-2, -3)

    row_full = valid_mb.view(1, 1, M, 1, B).expand(E, H, M, B, B)

    mk = torch.arange(M, device=q.device).view(1, M, 1)
    mq = torch.arange(M, device=q.device).view(1, 1, M)
    far_lower_refine = (mk <= mq - W_refine)

    deltas = []
    prev_ar = ar.clone()
    for _ in range(T - 1):
        al, _, cl = _local_pass(ar, kb, cr, sm_scale, row_full)
        l_hat = (al @ q_t.transpose(-1, -2)).to(ar.dtype) - cl[..., :, None]
        l_hat = l_hat + torch.where(far_lower_refine, 0.0, -float("inf"))
        row_max = torch.clamp(l_hat.max(dim=-2, keepdim=True).values, min=-1e30)
        row_max = torch.nan_to_num(row_max, neginf=0.0)
        exp_l = torch.nan_to_num(torch.exp(l_hat - row_max), nan=0.0)
        l = exp_l / (exp_l.sum(dim=-2, keepdim=True) + eps)
        cr = torch.sum(l, dim=-1).transpose(-1, -2)
        ar = (l.to(q_t.dtype) @ q_t).transpose(-2, -3)
        if return_deltas:
            deltas.append((ar - prev_ar).norm().item())
            prev_ar = ar.clone()

    al_full, y_full, cl_full = _local_pass_with_v(ar, kb, vb, cr, sm_scale, row_full)

    far_lower_t = (mk <= mq - W_blocks).transpose(-2, -1)
    l_hat_far = (q_t @ al_full.transpose(-1, -2)).to(q_t.dtype) - cl_full[..., None, :]
    l_hat_far = l_hat_far.masked_fill(~far_lower_t, -float("inf"))

    causal_local = torch.tril(torch.ones(B, B, dtype=torch.bool, device=q.device))
    outputs = []
    for m_q in range(M):
        w_start = max(0, m_q - W_blocks + 1)
        win_k = kb[:, :, w_start : m_q + 1].reshape(E, H, (m_q - w_start + 1) * B, D)
        win_v = vb[:, :, w_start : m_q + 1].reshape(E, H, (m_q - w_start + 1) * B, Dv)
        q_m = qb[:, :, m_q]

        n_win_blocks = m_q - w_start + 1
        win_valid = valid_mb[w_start : m_q + 1].reshape(-1)
        blk_idx = torch.arange(n_win_blocks, device=q.device).repeat_interleave(B)
        own_blk = n_win_blocks - 1
        intra = torch.arange(B, device=q.device).repeat(n_win_blocks)
        causal_win = (blk_idx < own_blk).unsqueeze(0) | (
            (blk_idx == own_blk).unsqueeze(0) & (intra.unsqueeze(0) <= torch.arange(B, device=q.device).unsqueeze(1))
        )
        win_mask = causal_win & win_valid.view(1, -1)

        local_scores = sm_scale * (q_m @ win_k.transpose(-1, -2))
        local_scores = local_scores.masked_fill(~win_mask.view(1, 1, B, -1), -float("inf"))

        far_logits_m = l_hat_far[:, :, :, m_q, :]
        combined = torch.cat([local_scores, far_logits_m], dim=-1)

        row_max = torch.clamp(combined.max(dim=-1, keepdim=True).values, min=-1e30)
        row_max = torch.nan_to_num(row_max, neginf=0.0)
        exp_combined = torch.exp(combined - row_max)
        exp_combined = torch.nan_to_num(exp_combined, nan=0.0)
        denom = exp_combined.sum(dim=-1, keepdim=True) + eps

        win_len = win_k.shape[-2]
        local_w = exp_combined[..., :win_len]
        far_w = exp_combined[..., win_len:]

        num_local = local_w @ win_v
        num_far = (far_w.unsqueeze(-1) * y_full).sum(dim=-2)

        out_m = (num_local + num_far) / denom
        outputs.append(out_m)

    z = torch.stack(outputs, dim=2).view(E, H, N_padded, Dv)
    z = z[..., :N, :]
    if return_deltas:
        return z, deltas
    return z

print("sliding_monarch_causal loaded")

In [ ]:
# ---- ported from ma_meta_bucket_route.py ----

def _kmeans_buckets(k_block, n_buckets, iters, sm_scale):
    Bl = k_block.shape[-2]
    n_buckets = min(n_buckets, Bl)
    idx0 = torch.linspace(0, Bl - 1, n_buckets, device=k_block.device).round().long().clamp(max=Bl - 1)
    centroids = k_block.index_select(-2, idx0).clone()
    assign = None
    for _ in range(iters):
        sims = sm_scale * (k_block @ centroids.transpose(-1, -2))
        assign = sims.argmax(dim=-1)
        onehot = F.one_hot(assign, n_buckets).to(k_block.dtype)
        sums = onehot.transpose(-1, -2) @ k_block
        counts = onehot.sum(dim=-2).unsqueeze(-1)
        new_centroids = sums / counts.clamp(min=1)
        centroids = torch.where(counts == 0, centroids, new_centroids)
    return centroids, assign


def monarch_meta_bucket_route(q, k, v, B, W_blocks, n_buckets, kmeans_iters=3, eps=1e-6):
    E, H, N, D = q.shape
    _, _, _, Dv = v.shape
    sm_scale = 1 / sqrt(D)

    M_base_needed = (N + B - 1) // B
    L = max(1, math.ceil(math.log2(max(M_base_needed, 2))))
    N_padded = B * (1 << L)
    pad = N_padded - N

    qb = F.pad(q, (0, 0, 0, pad)).view(E, H, -1, B, D)
    kb = F.pad(k, (0, 0, 0, pad)).view(E, H, -1, B, D)
    vb = F.pad(v, (0, 0, 0, pad)).view(E, H, -1, B, Dv)
    M = qb.shape[2]

    range_n = torch.arange(N_padded, device=q.device).view(M, B)
    valid_mb = range_n < N

    k_flat = F.pad(k, (0, 0, 0, pad))
    v_flat = F.pad(v, (0, 0, 0, pad))

    tier_centroids, tier_assign = [], []
    for l in range(L):
        Bl = B * (1 << l)
        Ml = N_padded // Bl
        k_l = k_flat.view(E, H, Ml, Bl, D)
        centroids, assign = _kmeans_buckets(k_l, n_buckets, kmeans_iters, sm_scale)
        tier_centroids.append(centroids)
        tier_assign.append(assign)

    outputs = []
    for m0 in range(M):
        q_m = qb[:, :, m0]

        w_start = max(0, m0 - W_blocks + 1)
        win_k = kb[:, :, w_start : m0 + 1].reshape(E, H, -1, D)
        win_v = vb[:, :, w_start : m0 + 1].reshape(E, H, -1, Dv)
        n_win_blocks = m0 - w_start + 1
        win_valid = valid_mb[w_start : m0 + 1].reshape(-1)
        blk_idx = torch.arange(n_win_blocks, device=q.device).repeat_interleave(B)
        own_blk = n_win_blocks - 1
        intra = torch.arange(B, device=q.device).repeat(n_win_blocks)
        causal_win = (blk_idx < own_blk).unsqueeze(0) | (
            (blk_idx == own_blk).unsqueeze(0) & (intra.unsqueeze(0) <= torch.arange(B, device=q.device).unsqueeze(1))
        )
        win_mask = causal_win & win_valid.view(1, -1)
        local_scores = sm_scale * (q_m @ win_k.transpose(-1, -2))
        local_scores = local_scores.masked_fill(~win_mask.view(1, 1, B, -1), -float("inf"))

        n = m0 - W_blocks + 1
        candidates = []
        if n > 0:
            for l in range(L):
                if (n >> l) & 1:
                    candidates.append((l, (n >> (l + 1)) << 1))

        tier_logits, tier_values = [], []
        for l, block_idx in candidates:
            Bl = B * (1 << l)
            Ml = N_padded // Bl
            if block_idx >= Ml:
                continue
            k_block = k_flat.view(E, H, Ml, Bl, D)[:, :, block_idx]
            v_block = v_flat.view(E, H, Ml, Bl, Dv)[:, :, block_idx]
            centroids = tier_centroids[l][:, :, block_idx]
            assign = tier_assign[l][:, :, block_idx]

            centroid_scores = sm_scale * (q_m @ centroids.transpose(-1, -2))
            chosen_bucket = centroid_scores.argmax(dim=-1)

            full_scores = sm_scale * (q_m @ k_block.transpose(-1, -2))
            member_mask = assign.unsqueeze(2) == chosen_bucket.unsqueeze(-1)
            scores = full_scores.masked_fill(~member_mask, -float("inf"))

            sub_max = scores.max(dim=-1, keepdim=True).values
            sub_max = torch.nan_to_num(sub_max, neginf=0.0)
            sub_exp = torch.nan_to_num(torch.exp(scores - sub_max), nan=0.0)
            level_logit = sub_max.squeeze(-1) + torch.log(sub_exp.sum(-1) + eps)
            sub_weights = sub_exp / (sub_exp.sum(-1, keepdim=True) + eps)
            level_value = sub_weights @ v_block
            tier_logits.append(level_logit)
            tier_values.append(level_value)

        if tier_logits:
            tier_logit_t = torch.stack(tier_logits, dim=-1)
            tier_value_t = torch.stack(tier_values, dim=-2)
            combined = torch.cat([local_scores, tier_logit_t], dim=-1)
        else:
            tier_value_t = None
            combined = local_scores

        row_max = torch.clamp(combined.max(dim=-1, keepdim=True).values, min=-1e30)
        row_max = torch.nan_to_num(row_max, neginf=0.0)
        exp_c = torch.nan_to_num(torch.exp(combined - row_max), nan=0.0)
        denom = exp_c.sum(dim=-1, keepdim=True) + eps

        win_len = win_k.shape[-2]
        local_w = exp_c[..., :win_len]
        num_local = local_w @ win_v

        if tier_logits:
            tier_w = exp_c[..., win_len:]
            num_tier = (tier_w.unsqueeze(-1) * tier_value_t).sum(dim=-2)
        else:
            num_tier = 0.0

        out_m = (num_local + num_tier) / denom
        outputs.append(out_m)

    z = torch.stack(outputs, dim=2).view(E, H, N_padded, Dv)
    return z[..., :N, :]

print("monarch_meta_bucket_route loaded")

In [ ]:
# shared needle-test helpers
D, Dv = 16, 16
BACKGROUND_NORM = 0.5 * (D ** 0.5)

def make_needle_scene(seed, N, needle_pos, key_scale=6.0):
    g = torch.Generator().manual_seed(seed)
    bq = (torch.randn(1, 1, N, D, generator=g) * 0.5).to(DEVICE)
    bk = (torch.randn(1, 1, N, D, generator=g) * 0.5).to(DEVICE)
    bv = (torch.randn(1, 1, N, Dv, generator=g) * 0.5).to(DEVICE)
    e = F.normalize(torch.randn(D, generator=g), dim=0).to(DEVICE)
    val = (F.normalize(torch.randn(Dv, generator=g), dim=0) * 5.0).to(DEVICE)
    k_full = bk.clone(); k_full[0, 0, needle_pos] = e * key_scale
    v_full = bv.clone(); v_full[0, 0, needle_pos] = val
    return bq, k_full, v_full, e, val

print("helpers loaded")

## Probe 1: Sinkhorn convergence characterization (is T convergence or propagation?)

T (refinement iterations) was the strongest lever found for SlidingMonarchAttention.
Mechanistically unexplained: is the T=1->8 improvement a *convergence* phenomenon
(per-iteration deltas shrink geometrically toward a fixed point) or a *propagation*
phenomenon (information takes multiple block-hops to reach the query)? Tests this by
tracking `||ar_t - ar_{t-1}||` per iteration, and separately checking whether the T
needed to recover a needle scales with the number of block-hops to it.

In [ ]:
B, W_blocks = 16, 4
N = 512

print("--- Per-iteration delta norms (does the refinement converge geometrically?) ---")
q = torch.randn(1, 1, N, D, device=DEVICE)
k = torch.randn(1, 1, N, D, device=DEVICE)
v = torch.randn(1, 1, N, Dv, device=DEVICE)
_, deltas = sliding_monarch_causal(q, k, v, B=B, W_blocks=W_blocks, T=10, return_deltas=True)
print("delta norms per iteration:", [f"{d:.4f}" for d in deltas])
if len(deltas) > 1:
    ratios = [deltas[i+1] / deltas[i] for i in range(len(deltas) - 1) if deltas[i] > 1e-9]
    print("consecutive ratios:", [f"{r:.3f}" for r in ratios])
    print("(ratio ~constant < 1 => geometric convergence; ratio far from constant => propagation-like)")

print()
print("--- Does required T scale with block-hop distance to the needle? ---")
distances_blocks = [1, 2, 4, 8, 16]
for T in (1, 2, 3, 4, 6, 8, 12):
    row = []
    for dist_blocks in distances_blocks:
        needle_pos = 18
        query_pos = needle_pos + dist_blocks * B + 20
        if query_pos >= N:
            row.append(None)
            continue
        bq, k_full, v_full, e, val = make_needle_scene(seed=1, N=N, needle_pos=needle_pos)
        q_full = bq.clone(); q_full[0, 0, query_pos] = e * 6.0
        z = sliding_monarch_causal(q_full, k_full, v_full, B=B, W_blocks=W_blocks, T=T)[0, 0, query_pos]
        row.append(F.cosine_similarity(z, val, dim=0).item())
    print(f"T={T:>3}: " + "  ".join(f"{r:.3f}" if r is not None else "  --" for r in row) +
          f"   (block-hops: {distances_blocks})")

## Probe 2: T x B surface

Does finer block size B need more T (more blocks -> more hops) or less (smaller
local problems converge faster)? Joint grid, not yet cross-tested tonight.

In [ ]:
N = 512
W_blocks = 4
Ts = (1, 2, 4, 8)
Bs = (8, 16, 32, 64)

print(f"{'B':>4} | " + " ".join(f"T={t:>3}" for t in Ts))
for B in Bs:
    row = []
    for T in Ts:
        cos_vals = []
        for trial in range(5):
            needle_pos = 18
            query_pos = min(N - 1, needle_pos + 14 * B)
            bq, k_full, v_full, e, val = make_needle_scene(seed=100 + trial, N=N, needle_pos=needle_pos)
            q_full = bq.clone(); q_full[0, 0, query_pos] = e * 6.0
            z = sliding_monarch_causal(q_full, k_full, v_full, B=B, W_blocks=W_blocks, T=T)[0, 0, query_pos]
            cos_vals.append(F.cosine_similarity(z, val, dim=0).item())
        row.append(sum(cos_vals) / len(cos_vals))
    print(f"{B:>4} | " + " ".join(f"{c:>5.3f}" for c in row))

## Probe 3: T x window width

Does a wide exact window reduce the T needed for the off-diagonal branch (window
handles near-field, T handles far-field -- independent?), or is there a real
interaction?

In [ ]:
N = 512
B = 16
Ts = (1, 2, 4, 8)
Ws = (1, 2, 4, 8)

print(f"{'W_blocks':>9} | " + " ".join(f"T={t:>3}" for t in Ts))
for W_blocks in Ws:
    row = []
    for T in Ts:
        cos_vals = []
        for trial in range(5):
            needle_pos = 18
            query_pos = min(N - 1, needle_pos + 14 * B)
            bq, k_full, v_full, e, val = make_needle_scene(seed=200 + trial, N=N, needle_pos=needle_pos)
            q_full = bq.clone(); q_full[0, 0, query_pos] = e * 6.0
            z = sliding_monarch_causal(q_full, k_full, v_full, B=B, W_blocks=W_blocks, T=T)[0, 0, query_pos]
            cos_vals.append(F.cosine_similarity(z, val, dim=0).item())
        row.append(sum(cos_vals) / len(cos_vals))
    print(f"{W_blocks:>9} | " + " ".join(f"{c:>5.3f}" for c in row))

## Probe 4: ragged N (not divisible by B)

Untested edge case guaranteed to occur in practice. Checks whether padding leaks
into the column-softmax normalization or biases representatives -- via the
identity-V causal-validity trick, at several N not divisible by B.

In [ ]:
B, W_blocks, T = 16, 2, 3
for N in (100, 127, 129, 200, 255, 257):
    q = torch.randn(1, 1, N, D, device=DEVICE)
    k = torch.randn(1, 1, N, D, device=DEVICE)
    eye = torch.eye(N, device=DEVICE).expand(1, 1, N, N)
    A = sliding_monarch_causal(q, k, eye, B=B, W_blocks=W_blocks, T=T)
    leak = torch.triu(A[0, 0], diagonal=1).abs().max().item()
    rs = A[0, 0].sum(-1)
    print(f"N={N:>4} (N%B={N%B:>2}): leak={leak:.2e}, rowsum min/max={rs.min().item():.6f}/{rs.max().item():.6f}")

## Probe 5: query-position boundary sawtooth

Recall as a function of the query's intra-block position. The first token of a
block has minimal local context, and the Fenwick-style tier decomposition changes
discontinuously at power-of-2-aligned positions (in the MetaMonarch/bucket-route
case) -- but even in plain SlidingMonarch, does recall show a sawtooth pattern
tied to intra-block position or block-boundary alignment?

In [ ]:
B, W_blocks, T = 16, 4, 3
N = 512
needle_pos = 18

print("Recall vs. intra-block offset of the query, fixed block-level distance:")
print(f"{'query_pos':>10} {'intra-block offset':>19} | {'cos':>7}")
base_query_block_start = needle_pos + 14 * B
for offset in range(0, B):
    query_pos = base_query_block_start + offset
    if query_pos >= N:
        continue
    bq, k_full, v_full, e, val = make_needle_scene(seed=1, N=N, needle_pos=needle_pos)
    q_full = bq.clone(); q_full[0, 0, query_pos] = e * 6.0
    z = sliding_monarch_causal(q_full, k_full, v_full, B=B, W_blocks=W_blocks, T=T)[0, 0, query_pos]
    cos = F.cosine_similarity(z, val, dim=0).item()
    print(f"{query_pos:>10} {offset:>19} | {cos:>7.4f}")

## Probe 6: adversarial low-norm, high-alignment needle

Every needle test tonight used a needle key with EITHER matched norm (same-norm
control) or boosted norm (original convention). This tries a needle with LOWER
norm than background (the opposite direction from the magnitude-artifact found in
the landmark-mechanics sweep) but still high query-alignment -- does fixed-scope
attention (window + Monarch far branch) still find it, or does low key-norm
suppress its own softmax weight regardless of alignment?

In [ ]:
B, W_blocks, T = 16, 4, 3
N = 256
needle_pos = 18
query_pos = 240

print(f"{'needle key scale':>18} {'(vs background ~2.0)':>22} | {'cos':>7}")
for key_scale in (2.0, 1.5, 1.0, 0.5, 0.25, 0.1):
    cos_vals = []
    for trial in range(5):
        bq, k_full, v_full, e, val = make_needle_scene(seed=300 + trial, N=N, needle_pos=needle_pos, key_scale=key_scale)
        q_full = bq.clone(); q_full[0, 0, query_pos] = e * 6.0  # query stays strongly aligned regardless
        z = sliding_monarch_causal(q_full, k_full, v_full, B=B, W_blocks=W_blocks, T=T)[0, 0, query_pos]
        cos_vals.append(F.cosine_similarity(z, val, dim=0).item())
    mean_c = sum(cos_vals) / len(cos_vals)
    print(f"{key_scale:>18.2f} {'':>22} | {mean_c:>7.4f}")

## Probe 7: bucket-routing capacity ceiling at larger scale

Extends the multi-needle finding (which showed routed buckets barely compete for
capacity) to larger blocks and more needles, at the scale a Colab GPU makes
comfortable but local CPU wasn't. How many simultaneous needles can be packed into
one block before EVEN routed exact attention saturates?

In [ ]:
B = 4
N = 512
QUERY_POS = 256
BLOCK_SPAN = 256  # bigger stressed block than the local-CPU version tested
W_blocks = 1

def make_multi_needle(seed, K):
    g = torch.Generator().manual_seed(seed)
    bq = (torch.randn(1, 1, N, D, generator=g) * 0.5).to(DEVICE)
    bk = (torch.randn(1, 1, N, D, generator=g) * 0.5).to(DEVICE)
    bv = (torch.randn(1, 1, N, Dv, generator=g) * 0.5).to(DEVICE)
    positions = torch.randperm(BLOCK_SPAN, generator=g)[:K].sort().values
    dirs = [F.normalize(torch.randn(D, generator=g), dim=0).to(DEVICE) for _ in range(K)]
    vals = [(F.normalize(torch.randn(Dv, generator=g), dim=0) * 5.0).to(DEVICE) for _ in range(K)]
    k_full, v_full = bk.clone(), bv.clone()
    for pos, e, val in zip(positions.tolist(), dirs, vals):
        k_full[0, 0, pos] = e * BACKGROUND_NORM
        v_full[0, 0, pos] = val
    return bq, k_full, v_full, dirs, vals

print(f"{'n_buckets':>10} {'K':>4} | {'mean cos':>9} {'min cos':>8} {'frac>0.5':>9}")
for n_buckets in (16, 64):
    for K in (1, 4, 16, 64, 128):
        bq, k_full, v_full, dirs, vals = make_multi_needle(seed=1, K=K)
        recalls = []
        for e, val in zip(dirs, vals):
            q_full = bq.clone()
            q_full[0, 0, QUERY_POS] = e * 6.0
            z = monarch_meta_bucket_route(q_full, k_full, v_full, B=B, W_blocks=W_blocks, n_buckets=n_buckets)[0, 0, QUERY_POS]
            recalls.append(F.cosine_similarity(z, val, dim=0).item())
        mean_r = sum(recalls) / len(recalls)
        min_r = min(recalls)
        frac_good = sum(1 for r in recalls if r > 0.5) / len(recalls)
        print(f"{n_buckets:>10} {K:>4} | {mean_r:>9.4f} {min_r:>8.4f} {frac_good:>9.2f}")
    print()

## Probe 8: hot-bucket load imbalance under non-uniform key distributions

Everything tonight used roughly isotropic Gaussian keys. Real trained-model key
distributions are anisotropic (a handful of directions carry most of the mass).
Simulates that by drawing keys from a low-rank-biased distribution and checking
how unevenly k-means routing distributes block membership across buckets --
relevant for real load-balancing on a 6-core CPU, per Fable's systems theme.

In [ ]:
B = 4
BLOCK_SPAN = 256
n_buckets = 16

def make_anisotropic_keys(seed, rank_bias):
    g = torch.Generator().manual_seed(seed)
    # low-rank-biased: most mass along `rank_bias` random directions, small isotropic noise on top
    basis = F.normalize(torch.randn(rank_bias, D, generator=g), dim=-1)
    coeffs = torch.randn(BLOCK_SPAN, rank_bias, generator=g)
    keys = coeffs @ basis + 0.15 * torch.randn(BLOCK_SPAN, D, generator=g)
    return (keys * 0.5).unsqueeze(0).unsqueeze(0).to(DEVICE)

print(f"{'rank_bias':>10} | {'bucket occupancy (min/mean/max)':>34} | {'imbalance ratio (max/mean)':>27}")
for rank_bias in (16, 4, 2, 1):
    k_block = make_anisotropic_keys(seed=1, rank_bias=rank_bias)
    sm_scale = 1 / (D ** 0.5)
    centroids, assign = _kmeans_buckets(k_block, n_buckets, 5, sm_scale)
    counts = torch.bincount(assign[0, 0], minlength=n_buckets).float()
    ratio = (counts.max() / counts.mean()).item()
    print(f"{rank_bias:>10} | min={counts.min().item():>4.0f} mean={counts.mean().item():>6.1f} max={counts.max().item():>4.0f}"
          f"{'':>10} | {ratio:>27.2f}")
print()
print("(rank_bias=16 ~ isotropic/uniform-ish, rank_bias=1 ~ maximally anisotropic;")
print(" higher imbalance ratio at low rank_bias would mean real trained-model key")
print(" distributions likely produce meaningfully hotter buckets than tonight's")
print(" Gaussian probes suggested -- relevant to the 6-core load-balancing question.)")

## Summary

Copy the output from each section above back for the research journal. No
verdicts are computed automatically here -- interpretation happens in the
follow-up conversation, same as every other probe tonight.